# trt_yolo_bench - Colab Runner

This notebook runs the repo pipeline in Colab from clone to scoring.
Run cells top-to-bottom.

## 1) Runtime checks
In Colab: Runtime -> Change runtime type -> GPU (T4 recommended).

In [ ]:
!nvidia-smi
!nvcc --version

## 2) Clone or enter repo

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/chao-dotcom/trt_yolo_bench.git"
REPO_DIR = Path("/content/trt_yolo_bench")

if not REPO_DIR.exists():
    !git clone {REPO_URL} /content/trt_yolo_bench

%cd /content/trt_yolo_bench
!git status --short

## 3) Install Python dependencies

In [ ]:
!python -m pip install -q --upgrade pip
!pip install -q ultralytics onnx pycocotools requests opencv-python numpy

# TensorRT / CUDA deps (best effort; exact package names may vary by Colab image).
!pip install -q tensorrt-cu12 || pip install -q tensorrt
!pip install -q pycuda

## 4) Prepare deterministic COCO slices + images

In [ ]:
!python export/download_coco_slice.py

## 5) Export ONNX

In [ ]:
!python export/export_onnx.py --weights yolo11n.pt --out-dir models

## 6) Build TensorRT engines (FP32/FP16/INT8)

In [ ]:
!python export/build_engines.py --onnx models/yolo11n.onnx --engines-dir engines

## 7) Build C++ harness

In [ ]:
%cd /content/trt_yolo_bench/harness
!cmake -S . -B build
!cmake --build build -j
%cd /content/trt_yolo_bench

## 8) Run harness (start with FP16)

In [ ]:
!./harness/build/trt_yolo_bench engines/yolo11n_fp16.engine data/val_slice.json --out results_fp16.json

## 9) Score COCO metrics

In [ ]:
!python eval/score_coco.py --dets results_fp16.json

## 10) Optional: run FP32 and INT8
Uncomment and run if engine files were built successfully.

In [ ]:
# !./harness/build/trt_yolo_bench engines/yolo11n_fp32.engine data/val_slice.json --out results_fp32.json
# !python eval/score_coco.py --dets results_fp32.json
# !./harness/build/trt_yolo_bench engines/yolo11n_int8.engine data/val_slice.json --out results_int8.json
# !python eval/score_coco.py --dets results_int8.json